# Experiment 6 — Structural authority probe with prefill control

Trains a per-(layer × position) structural probe on **matched-baseline** S-vs-U traces
with a **prefilled-response control**, then runs three follow-up evaluations:

1. **FAKE-delimiter transfer** — does fake `<|im_start|>system` text in the user field
   spoof the authority direction at the residual-stream level?
2. **NONE-conflict transfer** — does the structural probe correlate with `system_followed`
   on exp1b's held-out 200 conflicting pairs in NONE condition?
3. **Causal intervention sweep** — at the chosen best layer, add α·MM_direction to a
   U-condition forward pass and check whether behaviour flips toward S compliance.

Source dataset: `data/conflict_pairs_stratified_1k.jsonl` repurposed as a single-instruction
pool — each row contributes two items, using `s_instruction` and `u_instruction` separately
with the row's `original_prompt` as the natural follow-up message.

Difference-in-means (Marks & Tegmark 2024) is the **primary** probe; logistic regression
is fit alongside as a comparison so we can inspect MM-vs-LR cosine similarity per layer.

In [ ]:
# Install
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
!pip install -q -e {REPO_DIR}
!pip install -q vllm
os.environ["MECH_SPOOF_ROOT"] = REPO_DIR
src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()
import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
import os
from google.colab import drive, userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
drive.mount("/content/drive")

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = "qwen"
DRIVE_ROOT = Path("/content/drive/MyDrive/mech_spoof_results")

slug = MODEL_CONFIGS[MODEL_KEY].slug
OUT_DIR = DRIVE_ROOT / slug / "exp6_structural_authority"
OUT_DIR.mkdir(parents=True, exist_ok=True)

JUDGE_MODEL_ID = "Qwen/Qwen3.5-9B"   # same family as exp2b
BASELINE_SYSTEM = "You are a helpful assistant."

# Smoke knobs — set MAX_ROWS=20 for a fast end-to-end run before the real pass.
MAX_ROWS = None             # None = all ~1000 stratified-1k rows (→ ~2000 items)
DO_FAKE_TRANSFER = True
DO_NONE_CONFLICT = True
NONE_CONFLICT_N_PAIRS = 200
DO_INTERVENTION = True
INTERVENTION_N_PROMPTS = 50

print("OUT_DIR =", OUT_DIR)

## Run exp6 end-to-end

This single cell does:
free-gen R_S via vLLM → free vLLM → load HF model → extract activations at the three
response positions for all four trace conditions → fit MM + LR per (layer × position) →
compute MM-vs-LR cosine + multi-layer ensemble + prefilled→free transfer → FAKE eval →
intervention sweep → free HF → NONE-conflict generation + judging → save bundle.

Resumable: `OUT_DIR/free_responses.jsonl` is read on subsequent runs to skip vLLM
free-generation, and the activation cache (`OUT_DIR/act_cache_*`) is reused per condition.

In [ ]:
from mech_spoof.experiments import run_experiment_6

result = run_experiment_6(
    model_key=MODEL_KEY,
    out_dir=OUT_DIR,
    max_rows=MAX_ROWS,
    instruction_choices=("s", "u"),
    baseline_system=BASELINE_SYSTEM,
    do_fake_transfer=DO_FAKE_TRANSFER,
    do_none_conflict_transfer=DO_NONE_CONFLICT,
    none_conflict_n_pairs=NONE_CONFLICT_N_PAIRS,
    do_intervention_sweep=DO_INTERVENTION,
    intervention_n_prompts=INTERVENTION_N_PROMPTS,
    judge_model_id=JUDGE_MODEL_ID,
)
print(f"BEST: position={result.best_position} layer={result.best_layer} acc={result.best_accuracy:.3f}")
if result.fake_transfer:
    ft = result.fake_transfer
    print(
        f"FAKE: score mean={ft['fake_score_mean']:.3f}"
        f"  S_free={ft['s_free_score_mean']:.3f}  U_free={ft['u_free_score_mean']:.3f}"
    )
if result.none_conflict_transfer and result.none_conflict_transfer.get("pearson_r") is not None:
    nc = result.none_conflict_transfer
    print(f"NONE-conflict: r={nc['pearson_r']:.3f} p={nc['p_value']:.2e} n={nc['n_judgable']}")

## Recovery: judge-only (NONE-conflict generations crashed before judging)

Run if `OUT_DIR/none_conflict_generations.jsonl` exists but the judge step crashed.

In [ ]:
from mech_spoof.experiments import judge_generations_only_exp6

judge_generations_only_exp6(
    out_dir=OUT_DIR, model_key=MODEL_KEY, judge_model_id=JUDGE_MODEL_ID,
)

## Inspect: per-position × per-layer probe accuracy

Three curves per panel: MM val (prefilled), MM prefilled→free transfer (the diagnostic that
the probe reads structure rather than response content), and LR val. The right panel adds
the multi-layer-ensemble score as a single horizontal line.

In [ ]:
import json
import matplotlib.pyplot as plt

with open(OUT_DIR / "result.json") as f:
    rj = json.load(f)

n_layers = rj["n_layers"]
positions = list(rj["position_results"].keys())

fig, axes = plt.subplots(1, len(positions), figsize=(5 * len(positions), 4), sharey=True)
if len(positions) == 1:
    axes = [axes]
for ax, pos in zip(axes, positions):
    pr = rj["position_results"][pos]
    layers = list(range(n_layers))
    mm_val = [pr["mm_accuracies_val"][str(l)] for l in layers]
    mm_free = [pr["mm_accuracies_free_val"][str(l)] for l in layers]
    lr_val = [pr["lr_accuracies_val"][str(l)] for l in layers]
    ax.plot(layers, mm_val, marker=".", label="MM val (prefill)")
    ax.plot(layers, mm_free, marker=".", label="MM prefill→free")
    ax.plot(layers, lr_val, marker=".", alpha=0.6, label="LR val (prefill)")
    ax.axhline(pr["multi_layer_acc"], color="k", ls="--", lw=0.8,
               label=f"multi-L ensemble ({len(pr['multi_layer_top'])} layers)")
    ax.axhline(0.5, color="gray", ls=":", lw=0.5)
    ax.set_title(f"{pos}\nbest layer={pr['best_layer']} acc={pr['best_accuracy']:.3f}")
    ax.set_xlabel("layer"); ax.set_ylabel("accuracy")
    ax.legend(fontsize=8); ax.set_ylim(0.4, 1.02)
fig.suptitle(f"{MODEL_KEY}: structural probe accuracy per (position × layer)")
plt.tight_layout()
plt.savefig(OUT_DIR / "per_layer_accuracies.png", dpi=120)
plt.show()

## Inspect: MM vs LR cosine similarity per layer

High cosine (≥0.9) → LR is doing nothing extra; trust MM as the simpler/more causal probe.
Low cosine (≤0.7) → LR is exploiting a feature MM ignores; investigate.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for pos in positions:
    pr = rj["position_results"][pos]
    layers = sorted(int(k) for k in pr["mm_lr_cosine"])
    cos = [pr["mm_lr_cosine"][str(l)] for l in layers]
    ax.plot(layers, cos, marker=".", label=pos)
ax.axhline(0.9, color="green", ls="--", lw=0.7)
ax.axhline(0.7, color="orange", ls="--", lw=0.7)
ax.axhline(0.0, color="gray", ls=":", lw=0.5)
ax.set_xlabel("layer"); ax.set_ylabel("cos(MM, LR)")
ax.set_title(f"{MODEL_KEY}: MM vs LR direction agreement")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "mm_vs_lr_cosine.png", dpi=120)
plt.show()

## Inspect: FAKE-delimiter transfer

Distributions of probe scores on FAKE / S_free / U_free at the chosen best (layer, position).
If FAKE overlaps S_free, fake delimiters are spoofing the authority direction. If FAKE sits
with U_free, the model isn't fooled at the representation level.

In [ ]:
import numpy as np

ft = rj.get("fake_transfer")
if ft is None:
    print("No FAKE transfer in this run.")
else:
    fake = np.array(ft["fake_scores"])
    s_fr = np.array(ft["s_free_scores"])
    u_fr = np.array(ft["u_free_scores"])
    fig, ax = plt.subplots(figsize=(8, 4))
    bins = 40
    ax.hist(s_fr, bins=bins, alpha=0.5, label=f"S_free  μ={s_fr.mean():+.2f}", color="tab:blue")
    ax.hist(u_fr, bins=bins, alpha=0.5, label=f"U_free  μ={u_fr.mean():+.2f}", color="tab:red")
    ax.hist(fake, bins=bins, alpha=0.5, label=f"FAKE    μ={fake.mean():+.2f}", color="tab:green")
    ax.axvline(0, color="k", ls="--", lw=0.7)
    ax.set_xlabel("MM probe score (>0 = closer to S centroid)")
    ax.set_ylabel("count")
    ax.set_title(
        f"{MODEL_KEY}: FAKE-delim transfer @ {ft['position']} layer {ft['layer']}"
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "fake_transfer.png", dpi=120)
    plt.show()

## Inspect: NONE-conflict correlation

Probe score vs `system_followed` on the held-out 200 conflicting pairs in NONE condition.
A null result is the clean finding: structural authority encoding is separate from the
behavioural commitment computation.

In [ ]:
nc = rj.get("none_conflict_transfer")
if nc is None or nc.get("pearson_r") is None:
    print("No NONE-conflict transfer in this run.")
else:
    scores = np.array(nc["scores"])
    sf = np.array(nc["sys_followed"])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(scores[sf == 1], np.full(sf.sum(), 1.0) + np.random.uniform(-0.04, 0.04, sf.sum()),
               alpha=0.5, label="system_followed=1", color="tab:blue")
    ax.scatter(scores[sf == 0], np.full((sf == 0).sum(), 0.0) + np.random.uniform(-0.04, 0.04, (sf == 0).sum()),
               alpha=0.5, label="system_followed=0", color="tab:red")
    ax.axvline(0, color="k", ls="--", lw=0.7)
    ax.set_xlabel("MM probe score"); ax.set_ylabel("system_followed")
    ax.set_title(
        f"{MODEL_KEY}: NONE-conflict transfer @ {nc['position']} layer {nc['layer']}\n"
        f"r={nc['pearson_r']:.3f} p={nc['p_value']:.2e} n={nc['n_judgable']}"
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "none_conflict_transfer.png", dpi=120)
    plt.show()

## Inspect: causal intervention sweep

Sample generations at each α (in units of `||raw_direction||`). Eye-ball whether U-condition
behaviour shifts toward instruction-following at positive α.

In [ ]:
iv = rj.get("intervention")
if iv is None:
    print("No intervention sweep in this run.")
else:
    print(f"layer={iv['layer']}  natural_scale={iv['natural_scale']:.3f}  n_prompts={iv['n_prompts']}")
    for sw in iv["sweep"]:
        print(
            f"\nα = {sw['alpha']:+.3f} ({sw['alpha_in_natural_units']:+.2f} × natural):"
        )
        for k, sample in enumerate(sw["samples"]):
            preview = sample.replace("\n", " / ")[:200]
            print(f"  [{k}] {preview}")